# 08 — Forecast Accuracy & Value

Scores the published forecast against realised volumes and translates error into euros, per
[`../specifications/08-forecast-accuracy-value.md`](../specifications/08-forecast-accuracy-value.md).

**Depends on 07** (published net volume) and the per-leg silver forecasts (01, 03, 05, 06). Run last.

**Capability tables**
- `volume_forecast_gold_accuracy_daily` (MAE / RMSE / bias / MAPE / pinball by leg × zone × lead bucket)
- `volume_forecast_gold_model_comparison` (champion / challenger — MLflow story)
- `volume_forecast_gold_cost_of_error` (forecast error → cash-out €)
- `volume_forecast_gold_drift_alerts` (error-distribution drift monitoring)

Errors shrink as the lead time shortens (DA → RT) — the demo scales the residual by lead bucket so the
"closer to delivery, more accurate" story is visible.

**UC comments:** [`uc_table_comments.py`](./uc_table_comments.py) — applied in the final cell.

In [ ]:
import os
import datetime as dt

from pyspark.sql import functions as F

dbutils.widgets.text("catalog", os.environ.get("DEMO_UC_CATALOG", "energy_utilities"))
dbutils.widgets.text("schema", os.environ.get("DEMO_UC_SCHEMA", "energy_trading2"))

CATALOG = dbutils.widgets.get("catalog").strip() or "energy_utilities"
SCHEMA = dbutils.widgets.get("schema").strip() or "energy_trading2"
print(f"Target: {CATALOG}.{SCHEMA}")
spark.sql(f"USE `{CATALOG}`.`{SCHEMA}`")


def fq(name: str) -> str:
    return f"`{CATALOG}`.`{SCHEMA}`.`{name}`"


TODAY = dt.date.today()
NOW_TS = dt.datetime.combine(TODAY, dt.time(0, 0)) + dt.timedelta(minutes=15 * 56)  # ~14:00 cursor

REQUIRED = [
    "volume_forecast_gold_net_volume",
    "volume_forecast_silver_wind_forecast",
    "volume_forecast_silver_solar_forecast",
    "volume_forecast_silver_industrial_load",
    "volume_forecast_silver_consumption_st",
    "volume_forecast_silver_meter_profile",
]
missing = [t for t in REQUIRED if not spark.catalog.tableExists(fq(t))]
assert not missing, f"Run upstream notebooks first — missing: {missing}"

In [ ]:
# ---- Build a long-form (forecast vs actual) frame per leg, settled intervals only ----
def leg_frame(df, fc_col, act_col, leg):
    return (df.filter(F.col(act_col).isNotNull())
            .select("delivery_date", "zone_code",
                    F.col(fc_col).cast("double").alias("forecast_mw"),
                    F.col(act_col).cast("double").alias("actual_mw"),
                    F.lit(leg).alias("leg")))

wind = leg_frame(spark.table(fq("volume_forecast_silver_wind_forecast")), "p50_mw", "actual_mw", "WIND")
solar = leg_frame(spark.table(fq("volume_forecast_silver_solar_forecast")), "p50_mw", "actual_mw", "SOLAR")
ind = leg_frame(spark.table(fq("volume_forecast_silver_industrial_load")), "baseline_mw", "actual_mw", "INDUSTRIAL")

# Consumption: score the latest forecast vintage against its realised actual_mw (settled only).
_cons_all = spark.table(fq("volume_forecast_silver_consumption_st"))
_cons_latest_ts = _cons_all.agg(F.max("forecast_ts").alias("m")).first()["m"]
cons_latest = (_cons_all
               .filter(F.col("forecast_ts") == F.lit(_cons_latest_ts))
               .filter(F.col("actual_mw").isNotNull() & (F.col("interval_start") < F.lit(NOW_TS))))
cons = cons_latest.select(
    "delivery_date", "zone_code",
    F.col("p50_mw").alias("forecast_mw"),
    F.col("actual_mw").alias("actual_mw"),
    F.lit("CONSUMPTION").alias("leg"))

# NET: realised demand − realised supply at zone × interval (settled), vs published net.
k = ["delivery_date", "zone_code", "interval_start"]
w_i = spark.table(fq("volume_forecast_silver_wind_forecast")).filter(F.col("actual_mw").isNotNull()).groupBy(*k).agg(F.sum("actual_mw").alias("wind_act"))
s_i = spark.table(fq("volume_forecast_silver_solar_forecast")).filter(F.col("actual_mw").isNotNull()).groupBy(*k).agg(F.sum("actual_mw").alias("solar_act"))
i_i = spark.table(fq("volume_forecast_silver_industrial_load")).filter(F.col("actual_mw").isNotNull()).groupBy(*k).agg(F.sum("actual_mw").alias("ind_act"))
c_i = cons_latest.groupBy(*k).agg(F.sum("actual_mw").alias("cons_act"))
pub = (spark.table(fq("volume_forecast_gold_net_volume"))
       .filter(F.col("publication_status") == "PUBLISHED")
       .select(*k, F.col("net_volume_mw").alias("net_fc")))
net = (pub.join(w_i, k).join(s_i, k).join(i_i, k).join(c_i, k)
       .withColumn("forecast_mw", F.col("net_fc"))
       .withColumn("actual_mw", F.col("cons_act") + F.col("ind_act") - F.col("wind_act") - F.col("solar_act"))
       .select("delivery_date", "zone_code", "forecast_mw", "actual_mw", F.lit("NET").alias("leg")))

errs = wind.unionByName(solar).unionByName(ind).unionByName(cons).unionByName(net)

# ---- Daily error metrics per leg / zone, then split across lead buckets ----
agg = errs.groupBy("delivery_date", "leg", "zone_code").agg(
    F.avg("forecast_mw").alias("forecast_mw"),
    F.avg("actual_mw").alias("actual_mw"),
    F.avg(F.abs(F.col("forecast_mw") - F.col("actual_mw"))).alias("mae0"),
    F.sqrt(F.avg(F.pow(F.col("forecast_mw") - F.col("actual_mw"), 2))).alias("rmse0"),
    F.avg(F.col("forecast_mw") - F.col("actual_mw")).alias("bias0"),
    F.avg(F.abs(F.col("forecast_mw") - F.col("actual_mw")) / F.greatest(F.abs(F.col("actual_mw")), F.lit(1.0)) * 100.0).alias("mape0"),
)

buckets = spark.createDataFrame([("DA", 1.0), ("H+4", 0.7), ("H+1", 0.45), ("RT", 0.25)], ["lead_bucket", "f"])
accuracy = (agg.crossJoin(buckets).select(
    "delivery_date", "leg", "zone_code", "lead_bucket",
    F.round("forecast_mw", 2).alias("forecast_mw"),
    F.round("actual_mw", 2).alias("actual_mw"),
    F.round(F.col("mae0") * F.col("f"), 3).alias("mae_mw"),
    F.round(F.col("rmse0") * F.col("f"), 3).alias("rmse_mw"),
    F.round(F.col("bias0") * F.col("f"), 3).alias("bias_mw"),
    F.round(F.col("mape0") * F.col("f"), 2).alias("mape_pct"),
    F.round(F.col("mae0") * F.col("f") * 1.3, 3).alias("pinball_p90"),
))
accuracy.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq("volume_forecast_gold_accuracy_daily"))

print("accuracy_daily:", spark.table(fq("volume_forecast_gold_accuracy_daily")).count())
display(spark.table(fq("volume_forecast_gold_accuracy_daily"))
        .filter((F.col("leg") == "NET") & (F.col("delivery_date") == F.lit(TODAY)))
        .orderBy("zone_code", "lead_bucket"))

In [ ]:
# ---- Cost of error: NET forecast error translated to cash-out euros ----
PRICE_BY_BUCKET = {"DA": 60.0, "H+4": 80.0, "H+1": 110.0, "RT": 150.0}
price_expr = F.create_map(*[x for kv in PRICE_BY_BUCKET.items() for x in (F.lit(kv[0]), F.lit(kv[1]))])

net_acc = spark.table(fq("volume_forecast_gold_accuracy_daily")).filter(F.col("leg") == "NET")
cost = (net_acc
    .withColumn("imbalance_price_eur_mwh", price_expr[F.col("lead_bucket")])
    .withColumn("net_error_mw", F.col("bias_mw"))
    .withColumn("cashout_eur", F.round(F.abs(F.col("bias_mw")) * F.col("imbalance_price_eur_mwh") * 96.0 / 4.0, 2))
    .withColumn("avoidable_eur", F.round(F.col("cashout_eur") * 0.6, 2))
    .withColumn("value_at_1pct_mae_eur", F.round(F.abs(F.col("actual_mw")) * 0.01 * F.col("imbalance_price_eur_mwh") * 24.0, 2))
    .select("delivery_date", "zone_code", "lead_bucket",
            F.round("net_error_mw", 2).alias("net_error_mw"), "imbalance_price_eur_mwh",
            "cashout_eur", "avoidable_eur", "value_at_1pct_mae_eur"))
cost.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq("volume_forecast_gold_cost_of_error"))

# ---- Model comparison: champion vs challenger per leg (MLflow story) ----
from pyspark.sql import Row

champ_stats = {r["leg"]: (r["mae"], r["rmse"]) for r in (
    spark.table(fq("volume_forecast_gold_accuracy_daily"))
    .filter(F.col("lead_bucket") == "DA")
    .groupBy("leg").agg(F.avg("mae_mw").alias("mae"), F.avg("rmse_mw").alias("rmse")).collect())}

eval_start = TODAY - dt.timedelta(days=2)
mc_rows = []
for leg, (mae, rmse) in champ_stats.items():
    mae = float(mae or 0.0)
    rmse = float(rmse or 0.0)
    mc_rows.append(Row(model_id=f"{leg.lower()}_gbt_v3", leg=leg, role="CHAMPION",
                       eval_start_date=eval_start, eval_end_date=TODAY,
                       mae_mw=round(mae, 3), rmse_mw=round(rmse, 3),
                       skill_score=round(0.35 + 0.1, 3), is_promotion_candidate=False))
    mc_rows.append(Row(model_id=f"{leg.lower()}_lgbm_v4", leg=leg, role="CHALLENGER",
                       eval_start_date=eval_start, eval_end_date=TODAY,
                       mae_mw=round(mae * 0.9, 3), rmse_mw=round(rmse * 0.92, 3),
                       skill_score=round(0.35 + 0.18, 3), is_promotion_candidate=True))
spark.createDataFrame(mc_rows).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq("volume_forecast_gold_model_comparison"))

# ---- Drift alerts: error-distribution monitoring per leg × zone ----
import random
random.seed(8)
drift_base = (spark.table(fq("volume_forecast_gold_accuracy_daily"))
              .filter(F.col("lead_bucket") == "DA")
              .groupBy("leg", "zone_code").agg(F.avg("mae_mw").alias("current_mae")).collect())
drift_rows = []
for r in drift_base:
    current = float(r["current_mae"] or 0.0)
    baseline = current * random.uniform(0.7, 1.0)
    drift_pct = (current - baseline) / baseline * 100.0 if baseline > 0 else 0.0
    status = "DRIFT" if drift_pct > 25 else ("WATCH" if drift_pct > 12 else "STABLE")
    drift_rows.append(Row(detected_date=TODAY, leg=r["leg"], zone_code=r["zone_code"], metric="MAE",
                          baseline_value=round(baseline, 3), current_value=round(current, 3),
                          drift_pct=round(drift_pct, 1), drift_status=status))
spark.createDataFrame(drift_rows).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq("volume_forecast_gold_drift_alerts"))

display(spark.table(fq("volume_forecast_gold_cost_of_error")).orderBy(F.col("delivery_date").desc(), "zone_code", "lead_bucket"))

In [ ]:
# Row counts + Unity Catalog comments.
for t in [
    "volume_forecast_gold_accuracy_daily",
    "volume_forecast_gold_model_comparison",
    "volume_forecast_gold_cost_of_error",
    "volume_forecast_gold_drift_alerts",
]:
    print(f"  {t:44s}  {spark.table(fq(t)).count():>10,} rows")

from pathlib import Path

_uc_paths = []
try:
    _nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _uc_paths.append(Path(_nb).parent / "uc_table_comments.py")
except Exception:
    pass
_uc_paths.append(Path.cwd() / "uc_table_comments.py")

_uc_py = next((p for p in _uc_paths if p.is_file()), None)
if _uc_py is None:
    raise FileNotFoundError("uc_table_comments.py not found next to this notebook.")

exec(_uc_py.read_text(), globals())
apply_volume_forecast_notebook_08_comments(spark, CATALOG, SCHEMA)
print("UC comments applied for notebook 08.")